In [18]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Data Preprocessing and *Augmentation*

In [20]:
from torchvision import datasets, transforms

# Training images
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Validation images
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

Dataset Loading and Train/Validation Split

In [22]:
from pathlib import Path
from torchvision import datasets
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

raw = Path("/content/drive/MyDrive/ASL_Project/asl_dataset/asl_dataset")

train_data = datasets.ImageFolder(raw, transform=train_transform)
val_data = datasets.ImageFolder(raw, transform=val_transform)

assert train_data.classes == val_data.classes

class_names = train_data.classes
num_classes = len(class_names)

In [23]:
indices = list(range(len(train_data)))
targets = train_data.targets

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=targets
)

train_dataset = Subset(train_data, train_indices)
val_dataset = Subset(val_data, val_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

print("Class names:", class_names)
print("Number of classes:", num_classes)
print("Total images:", len(train_data))
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

Class names: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Number of classes: 36
Total images: 2515
Training images: 2012
Validation images: 503


ResNet18 Transfer Learning Model

In [24]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained ResNet18
model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

print("Device:", device)
print("Number of classes:", num_classes)
print("Class names:", class_names)

Device: cuda
Number of classes: 36
Class names: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


ResNet18 Model Train

In [25]:
import time

epochs = 3

epoch_times = []
train_losses = []
train_accuracies = []

total_start = time.time()

for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0
    epoch_start = time.time()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    accuracy = 100 * correct / total
    elapsed = time.time() - epoch_start

    train_losses.append(epoch_loss)
    train_accuracies.append(accuracy)
    epoch_times.append(elapsed)

    print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"Loss: {epoch_loss:.4f} "
        f"Accuracy: {accuracy:.2f}% "
        f"Time: {elapsed:.2f}s"
    )

Epoch [1/3] Loss: 1.9127 Accuracy: 66.35% Time: 867.10s
Epoch [2/3] Loss: 0.1536 Accuracy: 96.42% Time: 22.75s
Epoch [3/3] Loss: 0.0836 Accuracy: 97.66% Time: 22.81s


Model Evaluation

In [27]:
model.eval()

correct = 0
total = 0
val_loss = 0.0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

print("Validation Loss:", round(val_loss / total, 4))
print("Validation Accuracy:", round(100 * correct / total, 2), "%")

Validation Loss: 0.0874
Validation Accuracy: 97.22 %


Saving the model

In [30]:
from pathlib import Path
import torch

save_path = Path("/content/drive/MyDrive/ASL_Project/results")
save_path.mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "classes": class_names,
    "num_classes": num_classes
}, save_path / "restnet_asl.pth")

print("Model saved:", save_path / "restnet_asl.pth")
print("Class labels:", class_names)

Model saved: /content/drive/MyDrive/ASL_Project/results/restnet_asl.pth
Class labels: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
